# Mediciones en GPU · Voz a registro

Entorno de ejecución → Cambiar tipo → **GPU T4**. Luego Entorno de ejecución → **Ejecutar todas**. Tarda unos 40 minutos.

Antes: sube los 10 audios (`E01.mp3` … `L05.mp3`) a la carpeta `voz_a_registro_audio` de tu Google Drive.

## 1. ¿Hay GPU? (lo que dice el hardware, no la configuración)

In [ ]:
!nvidia-smi

## 2. Código, dependencias y audios

In [ ]:
!git clone -q https://github.com/BritneyMiranda/voz-a-registro.git 2>/dev/null || (cd voz-a-registro && git pull -q)
%cd /content/voz-a-registro
!pip install -q -r requirements-colab.txt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p audio && cp /content/drive/MyDrive/voz_a_registro_audio/*.mp3 audio/ && ls audio

ctranslate2 (el motor de faster-whisper) busca cuBLAS y cuDNN 9 por su cuenta: si no los encuentra, la corrida en GPU muere sin transcribir nada. Se instalan aquí y se agregan a `LD_LIBRARY_PATH`.

In [ ]:
!pip install -q nvidia-cublas-cu12 'nvidia-cudnn-cu12==9.*'
import os, nvidia.cublas.lib, nvidia.cudnn.lib
# son paquetes sin __init__.py: __file__ es None, la carpeta está en __path__
libs = [list(nvidia.cublas.lib.__path__)[0], list(nvidia.cudnn.lib.__path__)[0]]
os.environ['LD_LIBRARY_PATH'] = ':'.join(libs + [os.environ.get('LD_LIBRARY_PATH', '')])
print(os.environ['LD_LIBRARY_PATH'])

In [ ]:
import ctranslate2, torch
print('GPUs que ve ctranslate2:', ctranslate2.get_cuda_device_count())
print('torch.cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
# prueba corta en GPU: si falla, el error aparece aquí y no a mitad de la noche
!python -c "from faster_whisper import WhisperModel; import numpy as np; m = WhisperModel('Systran/faster-whisper-small', device='cuda', compute_type='float16'); s, i = m.transcribe(np.zeros(16000, np.float32)); list(s); print('GPU OK:', m.model.device)"

## 3. Escalera de degradación (L1 WhatsApp, L2 teléfono, L3 ruido)

In [ ]:
!python scripts/degradar.py

## 4. Transcripciones
Cada corrida es un proceso nuevo, así la VRAM medida incluye cargar el modelo.

In [ ]:
corridas = [
    '--modelo large-v3-turbo --precision float16 --dispositivo cuda',
    '--modelo large-v3-turbo --precision int8 --dispositivo cuda',
    '--modelo small --precision float16 --dispositivo cuda',
    '--modelo small --precision int8 --dispositivo cuda',
    '--modelo large-v3-turbo --precision float16 --dispositivo cuda --glosario hotwords',
    '--modelo large-v3-turbo --precision float16 --dispositivo cuda --glosario initial_prompt',
    '--modelo large-v3-turbo --precision float16 --dispositivo cuda --nivel L1',
    '--modelo large-v3-turbo --precision float16 --dispositivo cuda --nivel L2',
    '--modelo large-v3-turbo --precision float16 --dispositivo cuda --nivel L3',
    # CPU contra GPU en la MISMA máquina, sobre dos clips (en CPU es ~40 veces más lento)
    '--modelo large-v3-turbo --precision int8 --dispositivo cpu --clips E01 L01',
]
import subprocess
with open('cache/registro_colab.log', 'w') as log:
    for c in corridas:
        r = subprocess.run(f'python scripts/transcribir_corpus.py {c}', shell=True, capture_output=True, text=True)
        log.write(f'>>> {c}\n{r.stdout}\n{r.stderr}\n')
        print('>>>', c, '| OK' if r.returncode == 0 else '| FALLÓ')
        print(r.stdout[-600:] if r.returncode == 0 else r.stderr[-1500:])

## 5. Extracción con Qwen (modelo de lenguaje local, en la GPU)

In [ ]:
!python scripts/extraer_llm.py --corridas L0_large-v3-turbo_float16_cuda_glosario-ninguno L1_large-v3-turbo_float16_cuda_glosario-ninguno L2_large-v3-turbo_float16_cuda_glosario-ninguno L3_large-v3-turbo_float16_cuda_glosario-ninguno 2>&1 | tee -a cache/registro_colab.log

## 6. Métricas y descarga
Descarga `cache_colab.zip` y pásalo a la carpeta del proyecto.

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv > cache/gpu_colab.csv
!python scripts/evaluar.py
!zip -qr cache_colab.zip cache resultados
!cp cache_colab.zip /content/drive/MyDrive/voz_a_registro_audio/
from google.colab import files
files.download('cache_colab.zip')